In [3]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import pandas as pd
import numpy as np
import keras
from tensorflow.keras.models import Model
import tensorflow as tf
import numpy as np
import cv2
from astropy.io import fits
from astropy.convolution import convolve, Box1DKernel
import statistics
import matplotlib.pyplot as plt

project_dir =  Path(globals()['_dh'][0]).parent

current_dir =  Path(globals()['_dh'][0])
try:
    os.mkdir(os.path.join(current_dir, 'report_folder'))
except:
    print("Failed to create folder. Perhaps the folder already exists, or there's a permissions issue.")

Failed to create folder. Perhaps the folder already exists, or there's a permissions issue.


In [ ]:
# Load desired model for prediction and class map activation

model = keras.models.load_model(os.path.join(project_dir, r'models\best_model_originalUpdate.h5'))
# dom_model = keras.models.load_model(os.path.join(project_dir, r'models\rf_dom_v1.joblib'))

# we define a gradCAM class for loading the model and generating the heatmap.

class GradCAM:
    def __init__(self, model, classIdx, layerName=None):
        # store the model, the class index used to measure the class
        # activation map, and the layer to be used when visualizing
        # the class activation map
        self.model = model
        self.classIdx = classIdx
        self.layerName = layerName
        # if the layer name is None, attempt to automatically find
        # the target output layer
        if self.layerName is None:
            self.layerName = self.find_target_layer()


    def find_target_layer(self):
        # attempt to find the final convolutional layer in the network
        # by looping over the layers of the network in reverse order
        for layer in reversed(self.model.layers):
            # check to see if the layer has a 3D output
            if len(layer.output_shape) == 3:
                return layer.name
        # otherwise, we could not find a 4D layer so the GradCAM
        # algorithm cannot be applied
        raise ValueError("Could not find 3D layer. Cannot apply GradCAM.")


    def compute_heatmap(self, flux, eps=1e-8):
        # construct our gradient model by supplying (1) the inputs
        # to our pre-trained model, (2) the output of the (presumably)
        # final 3D layer in the network, and (3) the output of the
        # softmax activations from the model
        gradModel = Model(
            inputs=[self.model.inputs],
            outputs=[self.model.get_layer(self.layerName).output,
                     self.model.output])

        # record operations for automatic differentiation
        with tf.GradientTape() as tape:
            # cast the image tensor to a float-32 data type, pass the
            # image through the gradient model, and grab the loss
            # associated with the specific class index
            # inputs = tf.cast(image, tf.float32)
            (convOutputs, predictions) = gradModel(flux)
            loss = predictions[:, self.classIdx]
        # use automatic differentiation to compute the gradients
        grads = tape.gradient(loss, convOutputs)

        # compute the guided gradients
        castConvOutputs = tf.cast(convOutputs > 0, "float32")
        castGrads = tf.cast(grads > 0, "float32")
        guidedGrads = castConvOutputs * castGrads * grads
        # the convolution and guided gradients have a batch dimension
        # (which we don't need) so let's grab the volume itself and
        # discard the batch
        convOutputs = convOutputs[0]
        guidedGrads = guidedGrads[0]

        # compute the average of the gradient values, and using them
        # as weights, compute the ponderation of the filters with
        # respect to the weights
        weights = tf.reduce_mean(guidedGrads, axis=(0, 1))
        cam = tf.reduce_sum(tf.multiply(weights, convOutputs), axis=-1)

        # grab the spatial dimensions of the input image and resize
        # the output class activation map to match the input image
        # dimensions
        (w, h) = (flux.shape[2], flux.shape[1])
        heatmap = cv2.resize(cam.numpy(), (w, h))
        # normalize the heatmap such that all values lie in the range
        # [0, 1], scale the resulting values to the range [0, 255],
        # and then convert to an unsigned 8-bit integer
        numer = heatmap - np.min(heatmap)
        denom = (heatmap.max() - heatmap.min()) + eps
        heatmap = numer / denom
        heatmap = (heatmap * 255).astype("uint8")
        # return the resulting heatmap to the calling function
        return heatmap

    def overlay_heatmap(self, heatmap, image, alpha=0.5,
                        colormap=cv2.COLORMAP_VIRIDIS):
        # apply the supplied color map to the heatmap and then
        # overlay the heatmap on the input image
        heatmap = cv2.applyColorMap(heatmap, colormap)
        output = cv2.addWeighted(image, alpha, heatmap, 1 - alpha, 0)
        # return a 2-tuple of the color mapped heatmap and the output,
        # overlaid image
        return (heatmap, output)

In [4]:
ohe_dict = {'WDA': np.array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]),
            'WDZ': np.array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]),
            'WDB': np.array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]),
            'WDO': np.array([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]),
            'WD+MS': np.array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
            'WD': np.array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
            'sdX': np.array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
            'WDH': np.array([0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]),
            'WDELM': np.array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]),
            'WDC': np.array([0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]),
            'CV': np.array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
            'WDQ': np.array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])}

In [6]:
from email.mime import base


def extraction(fits_file):
    hdu  = fits.open(fits_file)
    data = hdu['COADD'].data
    w = 10**data['loglam']
    f = data['flux']*1e-17
    return w,f

def load_spectrum_data(file_path: str):
    if file_path.endswith('fits'):
        wavelength, flux = extraction(file_path)
    elif file_path.endswith('.dat'):
        sed = np.loadtxt(file_path, unpack = True)
        wavelength = sed[0,:]
        flux = sed[1,:]
    else:
        print("what format is that?")
        return None
    
    return wavelength, flux

base_wavelenght, _ = load_spectrum_data(os.path.join(project_dir, 'calibration_dat_file.dat'))
base_wavelenght = base_wavelenght[298:-600] # this is the wavelenght axis the model was trained on.

In [ ]:
# Now we can begin to process incoming data by preprocessing it.

filepath = str()
w, f = load_spectrum_data(os.path.join(project_dir, filepath))
# Interpolation
flux = np.interp(base_wavelenght, w, f)
# normalization
norm_magnitud = statistics.mean(flux[2050:2100])
flux = np.divide(flux, statistics.mean(flux[2050:2100]))

In [ ]:
# First we get the model prediction:



In [ ]:


fig, ax = plt.subplots(nrows = 3 ,figsize=(40,250))
index = 0
cam_wda = GradCAM(model, ohe_to_position(ohe_dict[class_for_map]))

for row in dz_spectrum_matrix[:30,:]:
    flux = row.reshape(1,-1,1)
    heatmap = cam_wda.compute_heatmap(flux)
    ax[index].plot(wavelength, row)
    sc = ax[index+1].scatter(wavelength, row, c=heatmap)
    #colorbar = fig.colorbar(sc, ax=ax[index + 1], pad=0.01)
    ax[index+2].plot(range(12), model.predict(row.reshape(1,-1,1)).reshape((12,)))
    ax[index+2].set_xticks(range(12))  
    ax[index+2].set_xticklabels(star_class)    
    index+=3